# Can we predict `passed_quiz`?
Goal: predict whether a student passes the quiz closing a session, from information available when the session ends.

**Design notes (checklist.md)**
- `data.csv` has no date/time column, so there is no "later rows" split. Students appear in several rows (about 7 sessions each), so both the test set and all CV folds hold **whole students** (grouped splits).
- `student_key` is an identifier and is dropped. `school_code` is a label, not a quantity, so it is one-hot encoded.
- Features are placement score (before the session), session length and hints used (both recorded during the session, before the closing quiz). If the model must predict before the session starts, `session_minutes` and `hints_used` would not be available; see the caveat at the end.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, log_loss
df = pd.read_csv('data.csv')
print(df.shape, 'students:', df.student_key.nunique(), 'pass rate: %.3f' % df.passed_quiz.mean())

## 1. Hold out the test set (whole students) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.student_key.nunique(), test.student_key.nunique())
# test is not touched again until the final scoring cell

## 2. Candidates, declared before any scoring
Target: `passed_quiz`. Primary metric: ROC AUC (threshold-free); the decision threshold is chosen afterwards on out-of-fold predictions.

| name | model | features |
|---|---|---|
| `baseline` | predicts training pass rate for everyone | none |
| `logit_A` | logistic regression, C=1 | prior_score, session_minutes, hints_used |
| `logit_B` | logistic regression, C=1 | A + enrol_year + school_code (one-hot) |
| `gbm_A` | hist. gradient boosting, depth 3, lr 0.05, 200 iters | A |
| `gbm_B` | hist. gradient boosting, same settings | B |

No other candidates or hyperparameters will be tried. Preprocessing (scaling, encoding) sits inside each pipeline so it is fitted within each CV fold.

In [ ]:
A = ['prior_score','session_minutes','hints_used']
B = A + ['enrol_year','school_code']
def make(kind, feats):
    num = [f for f in feats if f != 'school_code']
    cat = [f for f in feats if f == 'school_code']
    if kind == 'logit':
        pre = ColumnTransformer([('n', StandardScaler(), num)] + ([('c', OneHotEncoder(handle_unknown='ignore'), cat)] if cat else []))
        return Pipeline([('p', pre), ('m', LogisticRegression(C=1.0, max_iter=1000))])
    pre = ColumnTransformer([('n', 'passthrough', num)] + ([('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)] if cat else []))
    return Pipeline([('p', pre), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))])
candidates = {
 'baseline': (lambda: DummyClassifier(strategy='prior'), A),
 'logit_A': (lambda: make('logit', A), A), 'logit_B': (lambda: make('logit', B), B),
 'gbm_A': (lambda: make('gbm', A), A), 'gbm_B': (lambda: make('gbm', B), B)}

## 3. Grouped 5-fold cross-validation on training rows only

In [ ]:
gkf = GroupKFold(n_splits=5)
X, y, g = train, train.passed_quiz.values, train.student_key.values
folds = list(gkf.split(X, y, g))
oof = {}; rows = []
for name, (mk, feats) in candidates.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m = mk().fit(X.iloc[a][feats], y[a])
        p[b] = m.predict_proba(X.iloc[b][feats])[:, 1]
        aucs.append(roc_auc_score(y[b], p[b]))
    oof[name] = p
    rows.append((name, np.mean(aucs), np.std(aucs), log_loss(y, p)))
cv = pd.DataFrame(rows, columns=['model', 'cv_auc_mean', 'cv_auc_sd', 'oof_logloss']).set_index('model')
cv.round(4)

In [ ]:
# best model by mean CV AUC; threshold maximises balanced accuracy on its out-of-fold probabilities
best = cv.cv_auc_mean.idxmax()
ths = np.linspace(0.05, 0.95, 91)
bal = [balanced_accuracy_score(y, oof[best] >= t) for t in ths]
thr = float(ths[int(np.argmax(bal))])
print(best, 'threshold', round(thr, 2), 'OOF balanced acc %.4f' % max(bal))

## 4. Decision (written from CV results only, before the test set is scored)
- **Chosen model:** `logit_A` — logistic regression (C=1) on `prior_score`, `session_minutes`, `hints_used`. It has the highest mean grouped-CV AUC (about 0.604, fold SD about 0.008), ahead of the gradient-boosting models and above the 0.5 baseline. Adding `enrol_year` and `school_code` made both model types worse in CV, so they are left out.
- **Decision threshold:** 0.41, the value that maximises balanced accuracy on the out-of-fold probabilities (about 0.573).
- No further tuning. Next step: refit on all training rows and score the test set once.

## 5. Refit on all training rows, score the test set once (headline)

In [ ]:
final = candidates[best][0]().fit(train[A], y)
pt = final.predict_proba(test[A])[:, 1]
yt = test.passed_quiz.values
print('test AUC               : %.4f' % roc_auc_score(yt, pt))
print('test balanced acc @%.2f : %.4f' % (thr, balanced_accuracy_score(yt, pt >= thr)))
print('test log-loss          : %.4f (baseline using train pass rate: %.4f)' % (
    log_loss(yt, pt), log_loss(yt, np.full(len(yt), y.mean()))))
print('coefficients (standardised):', dict(zip(A, final.named_steps['m'].coef_[0].round(3))))

## 6. Conclusion and caveats
- Signal exists but is weak: AUC around 0.6 versus 0.5 for chance. Longer sessions go with passing and more hints go with failing; placement score adds little. This is a modest effect and not a basis for confident per-session predictions.
- The test set is about 360 students, so the headline AUC has a sampling uncertainty of roughly ±0.02; do not read differences of that size as meaningful.
- There is no timestamp, so the split is by student, not by time. It estimates performance on **new students**, not on future sessions of known students.
- `session_minutes` and `hints_used` are only known once the session has run, so this model predicts at the end of the session, just before the quiz. A pre-session prediction would have only `prior_score`, and would be weaker still.
- This is an association, not a causal claim (for example, students who struggle may open more hints).